<a href="https://colab.research.google.com/github/romeurf/Computational-Design-of-Graphene-Biosensors/blob/main/colab_boltz2_batch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Boltz-2 Batch — GFET Probe Structures
Corre o Boltz-2 em todas as probes do ZIP gerado pelo pipeline.  
Requer GPU: Runtime → Change runtime type → T4 GPU

In [4]:
# Célula 1 — Instalar Boltz-2
!pip install boltz -q
print("Boltz instalado.")

Boltz instalado.


In [5]:
# Célula 2 — Upload do ZIP e extracção dos YAMLs
import zipfile
from pathlib import Path
from google.colab import files

print("Faz upload do ficheiro boltz2_inputs.zip ...")
uploaded = files.upload()
zip_name = list(uploaded.keys())[0]

yaml_dir = Path("yamls")
yaml_dir.mkdir(exist_ok=True)
with zipfile.ZipFile(zip_name) as zf:
    zf.extractall(yaml_dir)

yaml_files = sorted(yaml_dir.glob("*.yaml"))
print(f"{len(yaml_files)} probes encontradas:")
for y in yaml_files:
    print(f"  {y.stem}")

Faz upload do ficheiro boltz2_inputs.zip ...


Saving boltz2_inputs.zip to boltz2_inputs (2).zip
25 probes encontradas:
  p1065_Spne_lytA_pos1134-1162_Tm66.5_GC57_hp-0.17_PASS
  p1073_Spne_lytA_pos1137-1163_Tm66.8_GC62_hp-0.17_PASS
  p1074_Spne_lytA_pos1137-1164_Tm67.6_GC59_hp-0.17_PASS
  p1075_Spne_lytA_pos1137-1165_Tm67.8_GC61_hp-0.17_PASS
  p1382_Spne_lytA_pos1281-1309_Tm67.1_GC57_hp-0.73_PASS
  p1877_Paer_oprL_pos255-283_Tm70.6_GC68_hp-1.08_PASS
  p205_Saur_nuc_pos366-394_Tm60.3_GC43_hp-0.38_PASS
  p214_Saur_nuc_pos369-395_Tm59.8_GC46_hp-0.38_PASS
  p215_Saur_nuc_pos369-396_Tm60.1_GC44_hp-0.38_PASS
  p216_Saur_nuc_pos369-397_Tm60.3_GC43_hp-0.38_PASS
  p2194_Paer_oprL_pos777-803_Tm70.6_GC69_hp-0.52_PASS
  p2195_Paer_oprL_pos777-804_Tm71.3_GC67_hp-0.52_PASS
  p2196_Paer_oprL_pos777-805_Tm71.5_GC68_hp-0.52_PASS
  p2205_Paer_oprL_pos780-806_Tm70.8_GC69_hp-0.96_PASS
  p3367_Hinf_frdB_pos225-253_Tm65.2_GC54_hp0.41_PASS
  p3597_Hinf_frdB_pos288-315_Tm64.5_GC52_hp-0.03_PASS
  p3598_Hinf_frdB_pos288-316_Tm64.8_GC54_hp-0.03_PASS
  p3630_

In [7]:
# Célula 3 — Correr Boltz-2 em todas as probes
import subprocess, json, csv

out_root = Path("boltz_results")
out_root.mkdir(exist_ok=True)
results = []

for i, yf in enumerate(yaml_files, 1):
    probe_id = yf.stem
    out_dir  = out_root / probe_id
    print(f"[{i}/{len(yaml_files)}] {probe_id}", flush=True)

    subprocess.run(
        ["boltz", "predict", str(yf),
         "--out_dir",          str(out_dir),
         "--recycling_steps",  "3",
         "--sampling_steps",   "200",
         "--diffusion_samples","1",
         "--accelerator",      "gpu",
         "--model",            "boltz2"],
        capture_output=True, text=True
    )

    conf_files = list(out_dir.glob("**/confidence_*.json")) or list(out_dir.glob("**/*.json"))
    confidence = ptm = plddt = None
    if conf_files:
        d = json.load(open(conf_files[0]))
        confidence = d.get("confidence_score") or d.get("confidence")
        ptm        = d.get("ptm")
        raw        = d.get("plddt")
        plddt      = round(sum(raw)/len(raw), 3) if isinstance(raw, list) else (d.get("plddt_score") or d.get("mean_plddt"))

    cif_files = list(out_dir.glob("**/*.cif"))
    status    = "OK" if cif_files else "FAILED"
    results.append({
        "probe_id":   probe_id,
        "status":     status,
        "confidence": round(confidence, 3) if confidence else "",
        "ptm":        round(ptm, 3)        if ptm        else "",
        "plddt":      round(plddt, 3)      if plddt      else "",
        "cif_path":   str(cif_files[0])    if cif_files  else "",
    })
    c = f"{confidence:.3f}" if confidence else "N/A"
    p = f"{ptm:.3f}"        if ptm        else "N/A"
    l = f"{plddt:.3f}"      if plddt      else "N/A"
    print(f"  {chr(10003) if status=="OK" else chr(10007)}  confidence={c}  pTM={p}  pLDDT={l}")

print(f"Concluido: {sum(1 for r in results if r["status"]=="OK")}/{len(results)} com sucesso.")

[1/25] p1065_Spne_lytA_pos1134-1162_Tm66.5_GC57_hp-0.17_PASS
  ✓  confidence=0.640  pTM=0.299  pLDDT=N/A
[2/25] p1073_Spne_lytA_pos1137-1163_Tm66.8_GC62_hp-0.17_PASS
  ✓  confidence=0.495  pTM=0.286  pLDDT=N/A
[3/25] p1074_Spne_lytA_pos1137-1164_Tm67.6_GC59_hp-0.17_PASS
  ✓  confidence=0.557  pTM=0.248  pLDDT=N/A
[4/25] p1075_Spne_lytA_pos1137-1165_Tm67.8_GC61_hp-0.17_PASS
  ✓  confidence=0.648  pTM=0.259  pLDDT=N/A
[5/25] p1382_Spne_lytA_pos1281-1309_Tm67.1_GC57_hp-0.73_PASS
  ✓  confidence=0.729  pTM=0.321  pLDDT=N/A
[6/25] p1877_Paer_oprL_pos255-283_Tm70.6_GC68_hp-1.08_PASS
  ✓  confidence=0.545  pTM=0.230  pLDDT=N/A
[7/25] p205_Saur_nuc_pos366-394_Tm60.3_GC43_hp-0.38_PASS
  ✓  confidence=0.508  pTM=0.259  pLDDT=N/A
[8/25] p214_Saur_nuc_pos369-395_Tm59.8_GC46_hp-0.38_PASS
  ✓  confidence=0.520  pTM=0.255  pLDDT=N/A
[9/25] p215_Saur_nuc_pos369-396_Tm60.1_GC44_hp-0.38_PASS
  ✓  confidence=0.510  pTM=0.260  pLDDT=N/A
[10/25] p216_Saur_nuc_pos369-397_Tm60.3_GC43_hp-0.38_PASS
  ✓  confid

In [11]:
import subprocess, json
import numpy as np

out_root = Path("boltz_results")
out_root.mkdir(exist_ok=True)
results = []

for i, yf in enumerate(yaml_files, 1):
    probe_id = yf.stem
    out_dir  = out_root / probe_id
    print(f"[{i}/{len(yaml_files)}] {probe_id}", flush=True)

    subprocess.run(
        ["boltz", "predict", str(yf),
         "--out_dir", str(out_dir),
         "--recycling_steps", "3",
         "--sampling_steps", "200",
         "--diffusion_samples", "1",
         "--accelerator", "gpu",
         "--model", "boltz2"],
        capture_output=True, text=True
    )

    # Confidence + pTM do JSON
    conf_files = list(out_dir.rglob("confidence_*_model_0.json"))
    confidence = ptm = None
    if conf_files:
        d = json.load(open(conf_files[0]))
        confidence = d.get("confidence_score") or d.get("confidence")
        ptm        = d.get("ptm")

    # pLDDT do ficheiro .npz separado
    plddt_files = list(out_dir.rglob("plddt_*_model_0.npz"))
    plddt = None
    if plddt_files:
        data  = np.load(plddt_files[0])
        arr   = data[data.files[0]]
        plddt = round(float(arr.mean()), 3)

    cif_files = list(out_dir.rglob("*_model_0.cif"))
    status = "OK" if cif_files else "FAILED"
    results.append({
        "probe_id":   probe_id,
        "status":     status,
        "confidence": round(confidence, 3) if confidence else "",
        "ptm":        round(ptm, 3)        if ptm        else "",
        "plddt":      plddt if plddt else "",
        "cif_path":   str(cif_files[0])    if cif_files  else "",
    })
    c = f"{confidence:.3f}" if confidence else "N/A"
    p = f"{ptm:.3f}"        if ptm        else "N/A"
    l = f"{plddt:.3f}"      if plddt      else "N/A"
    print(f"  conf={c}  pTM={p}  pLDDT={l}")

ok = sum(1 for r in results if r["status"] == "OK")
print(f"Concluido: {ok}/{len(results)}")

[1/25] p1065_Spne_lytA_pos1134-1162_Tm66.5_GC57_hp-0.17_PASS
  conf=0.640  pTM=0.299  pLDDT=0.726
[2/25] p1073_Spne_lytA_pos1137-1163_Tm66.8_GC62_hp-0.17_PASS
  conf=0.495  pTM=0.286  pLDDT=0.547
[3/25] p1074_Spne_lytA_pos1137-1164_Tm67.6_GC59_hp-0.17_PASS
  conf=0.557  pTM=0.248  pLDDT=0.635
[4/25] p1075_Spne_lytA_pos1137-1165_Tm67.8_GC61_hp-0.17_PASS
  conf=0.648  pTM=0.259  pLDDT=0.746
[5/25] p1382_Spne_lytA_pos1281-1309_Tm67.1_GC57_hp-0.73_PASS
  conf=0.729  pTM=0.321  pLDDT=0.830
[6/25] p1877_Paer_oprL_pos255-283_Tm70.6_GC68_hp-1.08_PASS
  conf=0.545  pTM=0.230  pLDDT=0.624
[7/25] p205_Saur_nuc_pos366-394_Tm60.3_GC43_hp-0.38_PASS
  conf=0.508  pTM=0.259  pLDDT=0.570
[8/25] p214_Saur_nuc_pos369-395_Tm59.8_GC46_hp-0.38_PASS
  conf=0.520  pTM=0.255  pLDDT=0.586
[9/25] p215_Saur_nuc_pos369-396_Tm60.1_GC44_hp-0.38_PASS
  conf=0.510  pTM=0.260  pLDDT=0.572
[10/25] p216_Saur_nuc_pos369-397_Tm60.3_GC43_hp-0.38_PASS
  conf=0.517  pTM=0.269  pLDDT=0.579
[11/25] p2194_Paer_oprL_pos777-803_Tm

In [12]:
!pip install py3Dmol -q
import py3Dmol
import numpy as np
from IPython.display import display, HTML

for r in results:
    if r["status"] != "OK" or not r["cif_path"]:
        continue
    cif_str = open(r["cif_path"]).read()
    view = py3Dmol.view(width=400, height=300)
    view.addModel(cif_str, "cif")
    view.setStyle({"cartoon": {"colorscheme": {"prop": "b", "gradient": "roygb", "min": 0.5, "max": 0.9}}})
    view.zoomTo()
    print(f"{r['probe_id']}  conf={r['confidence']}  pTM={r['ptm']}  pLDDT={r['plddt']}")
    view.show()

p1065_Spne_lytA_pos1134-1162_Tm66.5_GC57_hp-0.17_PASS  conf=0.64  pTM=0.299  pLDDT=0.726


3Dmol.js failed to load for some reason. Please check your browser console for error messages.

p1073_Spne_lytA_pos1137-1163_Tm66.8_GC62_hp-0.17_PASS  conf=0.495  pTM=0.286  pLDDT=0.547


3Dmol.js failed to load for some reason. Please check your browser console for error messages.

p1074_Spne_lytA_pos1137-1164_Tm67.6_GC59_hp-0.17_PASS  conf=0.557  pTM=0.248  pLDDT=0.635


...

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

p580_Nmen_rmpM_pos837-864_Tm60.4_GC44_hp0.16_PASS  conf=0.646  pTM=0.347  pLDDT=0.721


3Dmol.js failed to load for some reason. Please check your browser console for error messages.

p581_Nmen_rmpM_pos837-865_Tm61.1_GC46_hp0.16_PASS  conf=0.638  pTM=0.364  pLDDT=0.707


3Dmol.js failed to load for some reason. Please check your browser console for error messages.

In [1]:
import json, csv, shutil
from pathlib import Path
from google.colab import files

# Carregar resultados guardados
with open("results_backup.json") as f:
    results = json.load(f)

# Mostrar tabela
print(f"{'probe_id':<55} {'conf':>6} {'pTM':>6} {'pLDDT':>7} {'quality'}")
print("─" * 85)
for r in results:
    c = float(r["confidence"]) if r["confidence"] != "" else None
    q = "HIGH" if c and c >= 0.80 else ("MODERATE" if c and c >= 0.60 else ("LOW" if c else "N/A"))
    print(f"{r['probe_id']:<55} {str(r['confidence']):>6} {str(r['ptm']):>6} {str(r['plddt']):>7}  {q}")

# Guardar CSV
with open("boltz2_results_summary.csv", "w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=results[0].keys())
    w.writeheader()
    w.writerows(results)

# Zip dos CIFs + download
out_root = Path("boltz_results")
shutil.make_archive("boltz2_all_results", "zip", str(out_root))
files.download("boltz2_all_results.zip")
files.download("boltz2_results_summary.csv")

probe_id                                                  conf    pTM   pLDDT quality
─────────────────────────────────────────────────────────────────────────────────────
p1065_Spne_lytA_pos1134-1162_Tm66.5_GC57_hp-0.17_PASS     0.64  0.299   0.726  MODERATE
p1073_Spne_lytA_pos1137-1163_Tm66.8_GC62_hp-0.17_PASS    0.495  0.286   0.547  LOW
p1074_Spne_lytA_pos1137-1164_Tm67.6_GC59_hp-0.17_PASS    0.557  0.248   0.635  LOW
p1075_Spne_lytA_pos1137-1165_Tm67.8_GC61_hp-0.17_PASS    0.648  0.259   0.746  MODERATE
p1382_Spne_lytA_pos1281-1309_Tm67.1_GC57_hp-0.73_PASS    0.729  0.321    0.83  MODERATE
p1877_Paer_oprL_pos255-283_Tm70.6_GC68_hp-1.08_PASS      0.545   0.23   0.624  LOW
p205_Saur_nuc_pos366-394_Tm60.3_GC43_hp-0.38_PASS        0.508  0.259    0.57  LOW
p214_Saur_nuc_pos369-395_Tm59.8_GC46_hp-0.38_PASS         0.52  0.255   0.586  LOW
p215_Saur_nuc_pos369-396_Tm60.1_GC44_hp-0.38_PASS         0.51   0.26   0.572  LOW
p216_Saur_nuc_pos369-397_Tm60.3_GC43_hp-0.38_PASS        0.517  0.

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [13]:
# Célula 4 — Resumo dos resultados
import pandas as pd

df = pd.DataFrame(results)

def quality(row):
    c = row["confidence"]
    if c == "": return "N/A"
    c = float(c)
    if c >= 0.80: return "HIGH"
    if c >= 0.60: return "MODERATE"
    return "LOW"

df["quality"] = df.apply(quality, axis=1)
df.to_csv("boltz2_results_summary.csv", index=False)
print(df[["probe_id","status","confidence","ptm","plddt","quality"]].to_string(index=False))
print("CSV guardado: boltz2_results_summary.csv")

ValueError: numpy.dtype size changed, may indicate binary incompatibility. Expected 96 from C header, got 88 from PyObject

In [ ]:
# Célula 5 — Download dos resultados
import shutil
shutil.make_archive("boltz2_all_results", "zip", str(out_root))
files.download("boltz2_all_results.zip")
files.download("boltz2_results_summary.csv")
print("Download iniciado.")